# Tệp: notebooks/moocradar_colab.ipynb (Bản chạy riêng trong thư mục HTTT)
# Mục đích: Tái lập bước chuẩn hóa, huấn luyện, đánh giá và lộ trình học trên Google Colab.

Đặt repository và bốn file raw trên Google Drive. Kiểm tra các đường dẫn bên dưới rồi chạy cell theo thứ tự. Có thể chọn `bkt` hoặc `ncdm`; baseline luôn được giữ trong run. `smoke_test` dùng 25 sinh viên để kiểm tra nhanh, còn `full_run` dùng toàn bộ interaction. Artifacts và checkpoint được lưu trực tiếp trên Drive. Nên chạy notebook [Data Quality & EDA](01_data_quality_eda.ipynb) trước bước huấn luyện.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

# Source code từ GitHub
PROJECT_ROOT = Path("/content/Personalized-Learning-Path")

# Dữ liệu trên Google Drive
DATA_ROOT = Path("/content/drive/MyDrive/MOOC-Radar/data/raw")

# Dữ liệu xử lý và kết quả
PROCESSED_ROOT = PROJECT_ROOT / "data/processed"
ARTIFACT_ROOT = Path("/content/drive/MyDrive/MOOC-Radar/artifacts")

# Cấu hình NCDM
MODE = "full_run"
MODEL = "ncdm"
GRANULARITY = "middle"
HIDDEN_DIMS = "256,128"
DROPOUT = "0.5"

TRAIN = True
SAVED_RUN_DIR = None

# Kiểm tra
assert (PROJECT_ROOT / "requirements.txt").exists(), \
    "Không tìm thấy repository!"

assert (DATA_ROOT / "problem.json").exists(), \
    f"Không tìm thấy problem.json trong {DATA_ROOT}"

PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print("✅ GitHub:", PROJECT_ROOT)
print("✅ Data:", DATA_ROOT)
print("✅ Artifacts:", ARTIFACT_ROOT)

✅ GitHub: /content/Personalized-Learning-Path
✅ Data: /content/drive/MyDrive/MOOC-Radar/data/raw
✅ Artifacts: /content/drive/MyDrive/MOOC-Radar/artifacts


In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)
print("Đã cài đặt xong requirements.txt!")


CompletedProcess(args=['python', '-m', 'pip', 'install', '-r', '/content/Personalized-Learning-Path/requirements.txt'], returncode=0)

In [ ]:
import shutil
print("Số lõi CPU:", os.cpu_count())
print("RAM (GiB):", round(os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / 2**30, 2))
print("Dung lượng trống (GiB):", round(shutil.disk_usage("/content").free / 2**30, 2))
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có; pipeline hỗ trợ CPU")
except ImportError:
    print("GPU check không có; pipeline hỗ trợ CPU")


Số lõi CPU: 2
RAM (GiB): 12.67
Dung lượng trống (GiB): 87.2
GPU: không có; pipeline hỗ trợ CPU


## 1. Chạy kiểm tra chất lượng dữ liệu & EDA

Quét dữ liệu gốc, xuất báo cáo chất lượng và 8 biểu đồ tiếng Việt vào `artifacts/eda_<run_id>/`.


In [ ]:
env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT / "src"), "MPLCONFIGDIR": "/content/.mplconfig"}
eda_mode = "sample" if MODE == "smoke_test" else "full"
eda_cmd = ["python", "-m", "moocradar", "eda", "--data-root", str(DATA_ROOT), "--artifact-root", str(ARTIFACT_ROOT), "--mode", eda_mode]
res = subprocess.run(eda_cmd, cwd=PROJECT_ROOT, env=env, check=True, text=True, capture_output=True)
print(res.stdout)


## 2. Chuẩn bị dữ liệu (`prepare`)

Bước này đọc JSON array theo từng chuỗi, kiểm tra chất lượng và lưu các bảng chuẩn hóa (`problems.csv`, `concepts.csv`, `problem_concepts.csv`, `interactions.csv`).


In [ ]:
processed_dir = PROCESSED_ROOT / ("smoke" if MODE == "smoke_test" else "full")
prepare_cmd = ["python", "-m", "moocradar", "prepare", "--data-root", str(DATA_ROOT), "--output-dir", str(processed_dir)]
if MODE == "smoke_test":
    prepare_cmd += ["--max-students", "25"]
res = subprocess.run(prepare_cmd, cwd=PROJECT_ROOT, env=env, check=True, text=True, capture_output=True)
print(res.stdout)


CompletedProcess(args=['python', '-m', 'moocradar', 'prepare', '--data-root', '/content/drive/MyDrive/MOOC-Radar/data/raw', '--output-dir', '/content/Personalized-Learning-Path/data/processed/full'], returncode=0)

## 3. Huấn luyện, đánh giá và xuất hồ sơ/lộ trình (`train`)

Huấn luyện tạo thư mục run mới. NCDM tự chọn CUDA nếu có và fallback về CPU, in validation AUC/Brier mỗi epoch, early stopping và lưu `best_ncdm.pt`. Đặt `TRAIN=False` để load một run đã lưu trên Drive mà không train lại. BKT vẫn có checkpoint JSON và khả năng cập nhật tuần tự riêng.


In [ ]:
if TRAIN:
    before = set(ARTIFACT_ROOT.glob("*/manifest.json"))
    train_cmd = ["python", "-m", "moocradar", "train", "--model", MODEL, "--processed-dir", str(processed_dir), "--artifact-root", str(ARTIFACT_ROOT), "--seed", "42"]
    if MODEL == "ncdm":
        train_cmd += ["--granularity", GRANULARITY, "--batch-size", "512", "--learning-rate", "0.002", "--epochs", "3" if MODE == "smoke_test" else "20", "--patience", "4", "--hidden-dims", HIDDEN_DIMS, "--dropout", DROPOUT, "--device", "auto"]
    subprocess.run(train_cmd, cwd=PROJECT_ROOT, env=env, check=True)
    new_manifests = set(ARTIFACT_ROOT.glob("*/manifest.json")) - before
    assert len(new_manifests) == 1
    run_dir = next(iter(new_manifests)).parent
else:
    assert SAVED_RUN_DIR is not None, "Set SAVED_RUN_DIR when TRAIN=False"
    run_dir = Path(SAVED_RUN_DIR)
print("Run đã lưu:", run_dir)


Run đã lưu: /content/drive/MyDrive/MOOC-Radar/artifacts/20261020T160618189951Z_ncdm_3d00a8


In [ ]:
import csv
import json
import pandas as pd
from IPython.display import display

manifest = json.loads((run_dir / "manifest.json").read_text())
assert manifest["status"] == "complete"
with (run_dir / "metrics.csv").open() as f:
    for row in csv.DictReader(f):
        print(row)
required = ["data_quality.json", "predictions.csv", "metrics.csv", "student_mastery.csv", "cognitive_profile.csv", "knowledge_gaps.csv", "learning_paths.csv", "models/bkt.json"]
if MODEL == "ncdm":
    required += ["models/best_ncdm.pt", "models/id_mappings.json", "models/q_matrix.npz", "plots/training_loss.png", "plots/roc_curve.png"]
for name in required:
    assert (run_dir / name).exists(), name
print("Hồ sơ và lộ trình:", run_dir / "student_mastery.csv", run_dir / "learning_paths.csv")


{'model': 'bkt_frozen', 'split': 'test', 'n': '145307', 'threshold': '0.5', 'roc_auc': '0.514144', 'pr_auc': '0.809413', 'accuracy': '0.8029', 'balanced_accuracy': '0.505103', 'f1': '0.890336', 'precision': '0.804942', 'recall': '0.995999', 'specificity': '0.014206', 'brier': '0.158849', 'prevalence': '0.80332', 'positive_prediction_rate': '0.993992', 'tp': '116261', 'tn': '406', 'fp': '28173', 'fn': '467'}
{'model': 'bkt_frozen', 'split': 'validation', 'n': '128789', 'threshold': '0.5', 'roc_auc': '0.644992', 'pr_auc': '0.875678', 'accuracy': '0.821701', 'balanced_accuracy': '0.519298', 'f1': '0.901201', 'precision': '0.826212', 'recall': '0.991161', 'specificity': '0.047436', 'brier': '0.13944', 'prevalence': '0.820435', 'positive_prediction_rate': '0.98423', 'tp': '104729', 'tn': '1097', 'fp': '22029', 'fn': '934'}
{'model': 'bkt_online', 'split': 'test', 'n': '145307', 'threshold': '0.5', 'roc_auc': '0.781523', 'pr_auc': '0.925311', 'accuracy': '0.810333', 'balanced_accuracy': '0.5

## 4. Demo suy luận JSON cho 1 sinh viên (`demo`)

Chọn một sinh viên từ bảng interaction và load checkpoint để xuất JSON. BKT có thể nhận thêm một response để cập nhật trạng thái tuần tự. NCDM chuẩn chỉ suy luận từ checkpoint đóng băng; muốn hấp thụ dữ liệu mới phải retrain định kỳ. Tải cả thư mục run trên Drive để bàn giao cho dashboard.


In [ ]:
with (processed_dir / "interactions.csv").open() as f:
    user_id = next(csv.DictReader(f))["user_id"]
subprocess.run(["python", "-m", "moocradar", "demo", "--model", MODEL, "--run-dir", str(run_dir), "--processed-dir", str(processed_dir), "--user-id", user_id], cwd=PROJECT_ROOT, env=env, check=True)
print("Chia sẻ thư mục Drive này:", run_dir)


Chia sẻ thư mục Drive này: /content/drive/MyDrive/MOOC-Radar/artifacts/20261020T160618189951Z_ncdm_3d00a8
